# ARCHS4 saturation analysis at 50% sample coverage

**Environment:** `clamp-analyses`

For 50% sample coverage, runs CLAMPbase + CLAMPfull across 3 seeds and 6 K values derived as 5%, 10%, 25%, 50%, 75%, 100% of K_100 (CLAMP K from the 100% coverage model at seed 1). For each seed index, the SVD and FBM are loaded from the corresponding 06_bp_coverage_rshall output.

Output: `07_saturation/hall_saturation_rs50_k{K}_seed_{idx}/` per (K, seed) combination.


## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(Matrix)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [ ]:
base_output_dir <- config$ARCHS4$DATASET_FOLDER

coverage     <- 0.5
coverage_pct <- as.integer(coverage * 100)

pathways_path <- here::here('data/pathways')

N_CORES    <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_N_CORES
MULTIPLIER <- 100
MAX_ITER   <- 5000

base_seed <- config$ARCHS4$CLAMP_PARAMS$RANDOM_SVD_SEED  # 123
n_seeds   <- 3L

k_fracs <- c(0.05, 0.10, 0.25, 0.50, 0.75, 1.00)

output_dir <- file.path(base_output_dir, "07_saturation")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

message("Data coverage: ", coverage_pct, "% | K fracs: ", paste(k_fracs, collapse = ", "), " | seeds: 1-", n_seeds)


## Load preprocessed data and pathways

In [ ]:
meta_file <- file.path(base_output_dir, "metadata_filtered.rds")
if (!file.exists(meta_file)) {
  meta_file <- file.path(base_output_dir, "01_archs4_preprocess", "metadata_filtered.rds")
}
meta         <- readRDS(meta_file)
n_genes_thin <- meta$n_genes_thin
archs4_genes <- meta$gene_symbols_thin

hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list    <- list(HALL = hall_gmt, REACTOME = reactome_gmt, GOCC = gocc_gmt, C8 = c8_gmt)
all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, archs4_genes)
message("Loaded and matched all pathways | n_genes_thin = ", n_genes_thin)

## Run CLAMPbase + CLAMPfull for each seed and K value

In [ ]:
pct_to_subdir <- c(
  "50"  = "04_bp_coverage_hall_rs_50",
  "100" = "06_bp_coverage_hall_rs_100"
)

get_06_model_dir <- function(pct, seed_idx) {
  flat <- file.path(base_output_dir, sprintf("hall_coverage_rs%d_seed_%d", pct, seed_idx))
  if (file.exists(file.path(flat, "svd.rds"))) return(flat)
  file.path(base_output_dir, "06_bp_coverage_rshall",
            pct_to_subdir[as.character(pct)],
            sprintf("hall_coverage_rs%d_seed_%d", pct, seed_idx))
}

K_100    <- readRDS(file.path(get_06_model_dir(100L, 1L), "CLAMP_K.rds"))
k_values <- sort(unique(as.integer(round(k_fracs * K_100))))
message("K_100 = ", K_100, " | k_values = ", paste(k_values, collapse = ", "))


In [ ]:
for (seed_idx in seq_len(n_seeds)) {
  SEED <- base_seed + (seed_idx - 1L)
  message("\n", strrep("=", 60))
  message("seed_", seed_idx, " (SEED=", SEED, ") | Data coverage: ", coverage_pct, "%")
  message(strrep("=", 60))

  # Load pre-computed SVD and subsampled FBM from 06 output
  src_dir      <- get_06_model_dir(coverage_pct, seed_idx)
  svd_result   <- readRDS(file.path(src_dir, "svd.rds"))
  sub_info     <- readRDS(file.path(src_dir, "subsample_info.rds"))
  n_samples    <- sub_info$n_samples
  sample_names <- sub_info$sample_names
  Y_sub <- FBM(nrow = n_genes_thin, ncol = n_samples,
               backingfile = file.path(src_dir, "fbm_subsampled"), create_bk = FALSE)
  message("Loaded SVD and FBM: ", n_genes_thin, " x ", n_samples)

  for (CLAMP_K in k_values) {
    message("\n", strrep("-", 40))
    message("CLAMP_K = ", CLAMP_K)

    dst_dir <- file.path(output_dir,
      sprintf("hall_saturation_rs%d_k%d_seed_%d", coverage_pct, CLAMP_K, seed_idx))

    if (file.exists(file.path(dst_dir, "CLAMPfull_hall.rds"))) {
      message("Already done, skipping: ", dst_dir); next
    }

    dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)
    saveRDS(CLAMP_K, file.path(dst_dir, "CLAMP_K.rds"))
    saveRDS(list(
      seed = SEED, seed_idx = seed_idx, coverage = coverage,
      CLAMP_K = CLAMP_K, n_samples = n_samples
    ), file.path(dst_dir, "run_info.rds"))

    message("Running CLAMPbase (K = ", CLAMP_K, ")...")
    baseRes <- CLAMPbase(Y = Y_sub, svdres = svd_result, trace = TRUE, clamp_k = CLAMP_K)
    baseRes$Z <- data.frame(baseRes$Z); rownames(baseRes$Z) <- archs4_genes
    baseRes$B <- data.frame(baseRes$B); colnames(baseRes$B) <- sample_names
    saveRDS(baseRes, file.path(dst_dir, "CLAMPbase.rds"))
    model_dir <- file.path(dst_dir, "CLAMPbase")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(baseRes$B, file.path(model_dir, "B.csv"))
    write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

    message("Running CLAMPfull with all pathways prior...")
    fullRes <- CLAMPfull(
      Y = Y_sub, svdres = svd_result, priorMat = all_pathways_matched,
      clamp.base.result = baseRes, use_cpp = TRUE, trace = TRUE,
      multiplier = MULTIPLIER, max.iter = MAX_ITER, clamp_k = CLAMP_K
    )
    fullRes$Z <- data.frame(fullRes$Z); rownames(fullRes$Z) <- archs4_genes
    fullRes$B <- data.frame(fullRes$B); colnames(fullRes$B) <- sample_names
    fullRes$summary <- fullRes$summary %>%
      dplyr::rename(LV = LV_index) %>%
      dplyr::mutate(LV = paste0('LV', LV))
    saveRDS(fullRes, file.path(dst_dir, "CLAMPfull_hall.rds"))
    model_dir <- file.path(dst_dir, "CLAMPfull_hall")
    dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
    write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
    write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
    write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

    rm(baseRes, fullRes); gc()
    message("Done: ", dst_dir)
  }

  rm(Y_sub, svd_result); gc()
}

message("\n", strrep("=", 60))
message("All seeds and K values processed for ", coverage_pct, "% coverage!")
message(strrep("=", 60))
